# Notebook 4 — Test the Served Model

**Prerequisites:** Bob has deployed the ServingRuntime and InferenceService (Step 6).

**What you will do:**
1. Auto-detect the inference URL from your namespace.
2. Pre-process a test image into the format the model expects.
3. Send it to the KServe v2 inference endpoint and read the response.
4. Decode the class scores into a human-readable prediction.

In [ ]:
# ── Cell 0: Sync lab materials from GitHub ────────────────────────────────────
import subprocess, os, pathlib

REPO_URL   = 'https://github.com/faheemshai/1512_model_training.git'
LOCAL_PATH = os.path.expanduser('~/lab-materials')

if pathlib.Path(LOCAL_PATH, '.git').is_dir():
    r = subprocess.run(['git', '-C', LOCAL_PATH, 'pull', '--ff-only'],
                       capture_output=True, text=True)
    print('Repo up to date:', r.stdout.strip() or 'Already up to date.')
else:
    print('Cloning lab repo (first time, ~10 s)...')
    r = subprocess.run(['git', 'clone', REPO_URL, LOCAL_PATH],
                       capture_output=True, text=True)
    print(r.stderr.strip())

LAB = LOCAL_PATH
print(f'✅ Lab materials ready at: {LAB}')

In [ ]:
# ── Configuration — auto-detected from namespace ──────────────────────────────
import os

NAMESPACE      = open('/var/run/secrets/kubernetes.io/serviceaccount/namespace').read().strip()
MODEL_NAME     = 'image-classifier'
IMG_SIZE       = 64        # must match the size used during training
CLASSES        = ['intact', 'damaged', 'tampered']

# KServe HTTPRoute URL pattern on this cluster
INFERENCE_URL  = f'https://image-classifier-{NAMESPACE}.apps.itz-t53413.hub01-lb.techzone.ibm.com'
INFER_ENDPOINT = f'{INFERENCE_URL}/v2/models/{MODEL_NAME}/infer'

print(f'Namespace        : {NAMESPACE}')
print(f'Inference URL    : {INFERENCE_URL}')
print(f'Infer endpoint   : {INFER_ENDPOINT}')

In [ ]:
# ── Check the model is ready ─────────────────────────────────────────────────
import requests, warnings
warnings.filterwarnings('ignore')   # suppress SSL verify=False warning

ready_url = f'{INFERENCE_URL}/v2/models/{MODEL_NAME}/ready'
try:
    r = requests.get(ready_url, verify=False, timeout=15)
    if r.status_code == 200:
        print(f'✅ Model is Ready (HTTP {r.status_code})')
    else:
        print(f'⚠️  Model not ready yet (HTTP {r.status_code}) — wait 1–2 min and retry')
        print(f'   Response: {r.text[:200]}')
except Exception as e:
    print(f'❌ Could not reach inference service: {e}')
    print('   Check that the InferenceService is deployed and the URL is correct.')

In [ ]:
# ── Pre-process test image — uses local sample image from git repo ─────────────
import numpy as np, pathlib
from PIL import Image

# Use cardboard_box.jpg from the git repo — no internet needed
TEST_IMAGE_PATH = str(pathlib.Path(LAB) / 'sample-images' / 'cardboard_box.jpg')

# Pre-process: resize → RGB → CHW float32 normalised [0,1]
img = Image.open(TEST_IMAGE_PATH).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
arr = np.array(img, dtype=np.float32) / 255.0          # HWC
arr = np.transpose(arr, (2, 0, 1))                     # CHW
arr = np.expand_dims(arr, axis=0)                      # NCHW

print(f'Test image        : {TEST_IMAGE_PATH}')
print(f'Input tensor shape: {arr.shape}  (N, C, H, W)')
print(f'Value range       : [{arr.min():.2f}, {arr.max():.2f}]')

In [ ]:
# ── Send inference request ────────────────────────────────────────────────────
payload = {
    'inputs': [{
        'name'    : 'images',
        'shape'   : list(arr.shape),
        'datatype': 'FP32',
        'data'    : arr.flatten().tolist()
    }]
}

response = requests.post(
    INFER_ENDPOINT,
    json    = payload,
    headers = {'Content-Type': 'application/json'},
    verify  = False,
    timeout = 30
)

print(f'HTTP status: {response.status_code}')
if response.status_code != 200:
    print('Error body:', response.text[:500])
else:
    print('✅ Response received')

In [ ]:
# ── Decode the response ───────────────────────────────────────────────────────
resp_json  = response.json()
raw_scores = resp_json['outputs'][0]['data']

# Softmax: convert logits → probabilities
scores = np.array(raw_scores)
exp_s  = np.exp(scores - scores.max())
probs  = exp_s / exp_s.sum()

top_idx  = int(np.argmax(probs))
top_conf = float(probs[top_idx])

print('\n══════════════════════════════════════════')
print('  Classification Result')
print('══════════════════════════════════════════')
print(f'  Predicted class : {CLASSES[top_idx]}')
print(f'  Confidence      : {top_conf*100:.1f}%')
print('  All scores:')
for cls, prob in zip(CLASSES, probs):
    bar = '█' * int(prob * 30)
    print(f'    {cls:<12} {bar:<30} {prob*100:.1f}%')
print('══════════════════════════════════════════')
print('\n✅ Notebook 4 complete — proceed to Step 8 (deploy the app)')